# 05 - Early warning and audits
**Part A - Order-level risk tiers (DataCo, shipped orders).** The LightGBM score from notebook 04 is calibrated on the validation split and cut into
High / Medium / Low tiers. It is compared with a **5-row lookup table** (shipping mode, with Same Day split at the noon order cutoff).
**Part B - Random vs time split.** How much does a random split inflate the score?
**Part C - Audits of the two logistics files.** Evidence for why they are not used for modeling.

Run notebook 04 first: this notebook loads `models/late_delivery_lgbm.joblib`.

In [1]:
import sys; sys.path.append("..")
import yaml, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier, export_text
from lightgbm import LGBMClassifier

from src.data.load import load_dataco, load_logistics
from src.data.clean import clean_dataco, clean_logistics
from src.features.build_features import (
    DATACO_CAT, DATACO_NUM, DATACO_TARGET_CLF, build_dataco_features, make_preprocessor, shipped_only, time_split)
from src.models.evaluate import precision_at_recall, lift_at_top, pr_auc
from src.models.train_risk import fit_calibrator, assign_tiers, rule_key

cfg = yaml.safe_load(open("../config.yaml")); SEED = cfg["seed"]
tables = Path("../reports/tables"); tables.mkdir(parents=True, exist_ok=True)
Y = DATACO_TARGET_CLF

## Part A - risk tiers

In [2]:
bundle = joblib.load("../" + cfg["paths"]["models"] + "/late_delivery_lgbm.joblib")
model, FEATS = bundle["model"], bundle["features"]
full = build_dataco_features(clean_dataco(load_dataco("../" + cfg["paths"]["dataco"])))
data = shipped_only(full)
train, val, test = time_split(data, cfg["split"]["train"], cfg["split"]["val"])   # same split as notebook 04
s_val, s_test = model.predict_proba(val[FEATS])[:, 1], model.predict_proba(test[FEATS])[:, 1]

# calibrate on validation only, so that a score means a real late rate
cal = fit_calibrator(s_val, val[Y])
HI, LO = cfg["rules"]["tier_high_prob"], cfg["rules"]["tier_low_prob"]
test = test.assign(p_late=cal.predict(s_test))
test["tier"] = assign_tiers(test["p_late"], HI, LO)

order = ["High", "Medium", "Low"]
t = test.groupby("tier")[Y].agg(orders="size", late_rate="mean", late_orders="sum").reindex(order)
t["share_of_orders"] = t["orders"] / len(test)
t["share_of_late_orders"] = t["late_orders"] / test[Y].sum()
t = t.round(3); t.to_csv(tables / "early_warning_tiers.csv")
print(f"tiers: High >= {HI}, Low < {LO} (calibrated P(late)); base late rate on test = {test[Y].mean():.3f}")
t

tiers: High >= 0.7, Low < 0.2 (calibrated P(late)); base late rate on test = 0.576


,orders,late_rate,late_orders,share_of_orders,share_of_late_orders
tier,,,,,
High,9699,0.898,8714,0.374,0.584
Medium,15572,0.399,6212,0.601,0.416
Low,644,0.006,4,0.025,0.000


In [3]:
print("Tier by shipping mode (rows = orders):")
display(pd.crosstab(test["Shipping Mode"], test["tier"])[order])
sd = test[test["Shipping Mode"] == "Same Day"]
print("Same Day, by order hour (before / from noon):")
display(pd.crosstab(np.where(sd["order_hour"] >= 12, "from noon", "before noon"), sd["tier"]))
# is the calibrated probability honest?
bins = pd.cut(test["p_late"], [0, .2, .4, .6, .8, 1.0], include_lowest=True)
test.groupby(bins, observed=True).agg(orders=(Y, "size"), predicted=("p_late", "mean"), observed=(Y, "mean")).round(3)

Tier by shipping mode (rows = orders):


tier,High,Medium,Low
Shipping Mode,,,
First Class,3904,0,0
Same Day,810,0,635
Second Class,4985,162,0
Standard Class,0,15410,9


Same Day, by order hour (before / from noon):


tier,High,Low
row_0,,
before noon,0,635
from noon,810,0


,orders,predicted,observed
p_late,,,
"(-0.001, 0.2]",644,0.000,0.006
"(0.2, 0.4]",7927,0.387,0.389
"(0.4, 0.6]",7475,0.414,0.400
"(0.6, 0.8]",5135,0.786,0.803
"(0.8, 1.0]",4734,1.000,0.999


### Is the ML model just a lookup table?
The table below has 5 rows: one per shipping mode, with Same Day split at the noon cutoff. Its late rates are measured on the training split.

In [4]:
lookup = train.assign(key=rule_key(train)).groupby("key")[Y].agg(late_rate="mean", orders="size").round(3)
display(lookup)
p_rule = pd.Series(rule_key(test)).map(lookup["late_rate"]).to_numpy()
p_mode = test["Shipping Mode"].map(train.groupby("Shipping Mode")[Y].mean()).to_numpy()

def row(name, p):
    return {"model": name, "roc_auc": round(roc_auc_score(test[Y], p), 3), "pr_auc": round(pr_auc(test[Y], p), 3),
            "precision@80%recall": round(precision_at_recall(test[Y], p, 0.8), 3), "lift@top20%": round(lift_at_top(test[Y], p, 0.2), 2)}

cmp_ = pd.DataFrame([row("mode-only", p_mode), row("5-row lookup table", p_rule), row("LightGBM (notebook 04)", s_test)]).set_index("model")
cmp_.to_csv(tables / "lookup_vs_lightgbm.csv"); cmp_

,late_rate,orders
key,,
First Class,1.000,18744
Same Day (before noon),0.000,3506
Same Day (from noon),1.000,2969
Second Class,0.801,23682
Standard Class,0.399,72034


,roc_auc,pr_auc,precision@80%recall,lift@top20%
model,,,,
mode-only,0.746,0.84,0.653,1.65
5-row lookup table,0.777,0.86,0.677,1.70
LightGBM (notebook 04),0.779,0.86,0.677,1.71


**How to read this.** If the lookup table matches LightGBM, the model contributes nothing a business rule does not. That is the finding:
lateness in DataCo is set by the promised shipping time and the noon cutoff, and the rest is random.
If LightGBM is still clearly ahead, something besides mode and noon matters, and the next step is to find what.

## Part B - random split vs time split
If a random split gives a higher score, the evaluation was inflated. DataCo rows are order *items*: items of one order share the same mode and delivery time, so a random split can put near-duplicates on both sides.

In [5]:
bp = dict(num_leaves=31, learning_rate=0.1, n_estimators=200, min_child_samples=100)
FIXED = dict(subsample=0.8, subsample_freq=1, colsample_bytree=0.8, class_weight="balanced", random_state=SEED, verbose=-1)
cat = [c for c in FEATS if c in DATACO_CAT]; num = [c for c in FEATS if c in DATACO_NUM]

def fit_auc(tr, te):
    m = Pipeline([("prep", make_preprocessor(cat, num)), ("clf", LGBMClassifier(**bp, **FIXED))]).fit(tr[FEATS], tr[Y])
    return round(roc_auc_score(te[Y], m.predict_proba(te[FEATS])[:, 1]), 3)

n = len(data); idx = np.random.RandomState(SEED).permutation(n)
rand_tr, rand_te = data.iloc[idx[:int(.70 * n)]], data.iloc[idx[int(.85 * n):]]
res = pd.DataFrame({"test ROC-AUC": {"time split (oldest -> newest)": fit_auc(train, time_split(data, cfg["split"]["train"], cfg["split"]["val"])[2]),
                                     "random split": fit_auc(rand_tr, rand_te)}})
if "Order Id" in data.columns:
    print(f"Share of rows whose order has several items: {data['Order Id'].duplicated(keep=False).mean():.3f}")
res

Share of rows whose order has several items: 0.890


,test ROC-AUC
time split (oldest -> newest),0.779
random split,0.801


## Part C - audits of the two logistics files
Both files were checked for (1) whether the target can be rebuilt from other columns and (2) whether the remaining columns predict it at all.

In [6]:
audit = []
def add(dataset, check, result): audit.append({"dataset": dataset, "check": check, "result": result})

# ---- dynamic_supply_chain_logistics_dataset
lg = clean_logistics(load_logistics("../" + cfg["paths"]["logistics"]))
num_lg = lg.select_dtypes("number")
TG = ["delivery_time_deviation", "delay_probability", "disruption_likelihood_score"]
raw_f = [c for c in num_lg.columns if c not in TG]
D = "dynamic logistics (32k rows)"
add(D, "largest |corr| between any raw column and any target", f"{num_lg.corr()[TG].drop(TG).abs().values.max():.3f}")
add(D, "lag-1 autocorrelation of traffic / weather / delay (real series are high)",
    ", ".join(f"{c.split('_')[0]} {lg[c].autocorr(1):+.3f}" for c in ["traffic_congestion_level", "weather_condition_severity", "delay_probability"]))
rule = pd.cut(lg["disruption_likelihood_score"], [-np.inf, .3, .7, np.inf], labels=["Low Risk", "Moderate Risk", "High Risk"], right=False)
add(D, "risk_classification == bins of disruption_likelihood_score (<0.3, <0.7, else)", f"{(rule.astype(str) == lg['risk_classification']).mean():.1%} agreement")
cut = int(len(lg) * .85); tr, te = lg.iloc[:cut], lg.iloc[cut:]
for tg in TG:
    m = RandomForestRegressor(100, min_samples_leaf=10, n_jobs=-1, random_state=SEED).fit(tr[raw_f], tr[tg])
    add(D, f"time-split R2 for {tg} from raw columns", f"{r2_score(te[tg], m.predict(te[raw_f])):+.3f}")
maj = tr["risk_classification"].mode()[0]
add(D, "risk class accuracy: always '" + maj + "'", f"{(te['risk_classification'] == maj).mean():.3f}")
for name, cols in [("raw columns only", raw_f), ("raw + disruption_likelihood_score (leak)", raw_f + ["disruption_likelihood_score"])]:
    m = RandomForestClassifier(100, min_samples_leaf=10, n_jobs=-1, random_state=SEED).fit(tr[cols], tr["risk_classification"])
    add(D, f"risk class accuracy, {name}", f"{accuracy_score(te['risk_classification'], m.predict(te[cols])):.3f}")

# ---- smart_logistics_dataset
sm = pd.read_csv("../" + cfg["paths"]["smart_logistics"]); y = sm["Logistics_Delay"]; S = "smart logistics (1k rows)"
rule = ((sm["Shipment_Status"] == "Delayed") | (sm["Traffic_Status"] == "Heavy")).astype(int)
add(S, "Logistics_Delay == (Shipment_Status is Delayed) OR (Traffic_Status is Heavy)", f"{(rule == y).mean():.1%} agreement")
X_all = pd.get_dummies(sm.drop(columns=["Timestamp", "Logistics_Delay", "Logistics_Delay_Reason"]),
                       columns=["Asset_ID", "Shipment_Status", "Traffic_Status"]).astype(float)
X_clean = X_all.drop(columns=[c for c in X_all if c.startswith(("Shipment_Status", "Traffic_Status"))])
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rf = RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
add(S, "5-fold ROC-AUC, all columns (leak)", f"{cross_val_score(rf, X_all, y, cv=cv, scoring='roc_auc').mean():.3f}")
add(S, "5-fold ROC-AUC without Shipment_Status and Traffic_Status", f"{cross_val_score(rf, X_clean, y, cv=cv, scoring='roc_auc').mean():.3f}")
add(S, "Logistics_Delay_Reason vs delay (chi-square p-value; reason is filled in for on-time rows too)",
    f"{chi2_contingency(pd.crosstab(sm['Logistics_Delay_Reason'].fillna('none'), y))[1]:.3f}")
add(S, "Asset_ID vs delay (chi-square p-value)", f"{chi2_contingency(pd.crosstab(sm['Asset_ID'], y))[1]:.3f}")
add(S, "numeric columns: largest |corr| with delay", f"{sm.select_dtypes('number').drop(columns='Logistics_Delay').corrwith(y).abs().max():.3f}")

audit_df = pd.DataFrame(audit); audit_df.to_csv(tables / "logistics_audits.csv", index=False)
pd.set_option("display.max_colwidth", 120); audit_df

,dataset,check,result
0,dynamic logistics (32k rows),largest |corr| between any raw column and any target,0.014
1,dynamic logistics (32k rows),lag-1 autocorrelation of traffic / weather / delay (real series are high),"traffic +0.003, weather +0.001, delay +0.006"
2,dynamic logistics (32k rows),"risk_classification == bins of disruption_likelihood_score (<0.3, <0.7, else)",100.0% agreement
3,dynamic logistics (32k rows),time-split R2 for delivery_time_deviation from raw columns,-0.004
4,dynamic logistics (32k rows),time-split R2 for delay_probability from raw columns,-0.015
5,dynamic logistics (32k rows),time-split R2 for disruption_likelihood_score from raw columns,-0.011
6,dynamic logistics (32k rows),risk class accuracy: always 'High Risk',0.760
7,dynamic logistics (32k rows),"risk class accuracy, raw columns only",0.760
8,dynamic logistics (32k rows),"risk class accuracy, raw + disruption_likelihood_score (leak)",1.000
9,smart logistics (1k rows),Logistics_Delay == (Shipment_Status is Delayed) OR (Traffic_Status is Heavy),100.0% agreement


## Results to copy into `reports/experiment_log.md`
Tiers (shipped orders, test): High ... % of orders, late rate ...; Medium ...; Low ...
Lookup table vs LightGBM: ROC-AUC ... vs ...; precision@80%recall ... vs ...
Time vs random split: ... vs ...

## Takeaways (fill in)
1. 
2. 
3. 